# Model Comparison: GSF 2017 vs 2025

The 2025 update extends the input dataset and refits the splines, changing both central fluxes and uncertainties. We compare the two versions for `GSFEnergy` and `GSFEnergyPerNucleon`, scaled by $E^{2.7}$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from globalsplinefit import GSFEnergy, GSFEnergyPerNucleon

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 11
plt.rcParams["lines.linewidth"] = 1.2
plt.rcParams["grid.alpha"] = 0.3

In [ ]:
energy = np.logspace(np.log10(0.5), np.log10(1e11), 500)
energy_per_nucleon = energy

models_e = {v: GSFEnergy(version=v) for v in ("2017", "2025")}
models_n = {v: GSFEnergyPerNucleon(version=v) for v in ("2017", "2025")}

groups = ["p", "He", "O*", "Fe*"]
group_labels = ["Proton", "Helium", "Oxygen*", "Iron*"]
group_colors = ["red", "orange", "green", "blue"]
line_styles = {"2025": "-", "2017": "--"}
energy_exponent = 2.7


def collect(models, grid):
    flux = {v: {g: m.flux(grid, g) for g in groups} for v, m in models.items()}
    err = {v: {g: m.error(grid, g) for g in groups} for v, m in models.items()}
    for v, m in models.items():
        flux[v]["Total"] = m.total_flux(grid)
        err[v]["Total"] = m.total_error(grid)
    return flux, err


energy_fluxes, energy_errors = collect(models_e, energy)
nucleon_fluxes, nucleon_errors = collect(models_n, energy_per_nucleon)

## Total-energy flux comparison

Solid: 2025. Dashed: 2017.

1. Per-group + total flux.
2. Relative uncertainty $\sigma/\Phi$.
3. Flux with 1-sigma error bands.

In [ ]:
fig, ax = plt.subplots()

for group, label, color in zip(groups, group_labels, group_colors, strict=False):
    for v in ("2017", "2025"):
        flux = energy_fluxes[v][group]
        ax.loglog(
            energy,
            flux * energy**energy_exponent,
            ls=line_styles[v],
            color=color,
            alpha=0.8,
            label=f"{label} ({v})" if v == "2025" else None,
        )

for v in ("2017", "2025"):
    flux = energy_fluxes[v]["Total"]
    ax.loglog(
        energy,
        flux * energy**energy_exponent,
        ls=line_styles[v],
        color="black",
        lw=2,
        alpha=0.9,
        label=f"Total ({v})",
    )

ax.set_xlabel("Energy [GeV]")
ax.set_ylabel(f"Flux × $E^{{{energy_exponent}}}$")
ax.grid(True, alpha=0.3)
ax.legend(loc="lower center", ncol=2)
ax.set_xlim(energy[0], energy[-1])
ax.set_ylim(8.0, 9e4)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()

for group, label, color in zip(groups, group_labels, group_colors, strict=False):
    for v in ("2017", "2025"):
        rel = energy_errors[v][group] / energy_fluxes[v][group]
        ax.loglog(
            energy,
            rel,
            ls=line_styles[v],
            color=color,
            alpha=0.8,
            label=f"{label} ({v})" if v == "2025" else None,
        )

for v in ("2017", "2025"):
    rel = energy_errors[v]["Total"] / energy_fluxes[v]["Total"]
    ax.loglog(
        energy,
        rel,
        ls=line_styles[v],
        color="black",
        lw=2,
        alpha=0.9,
        label=f"Total ({v})",
    )

ax.set_xlabel("Energy [GeV]")
ax.set_ylabel("Relative error σ/Φ")
ax.grid(True, alpha=0.3)
ax.legend(loc="lower right", ncol=2)
ax.set_xlim(energy[0], energy[-1])
ax.set_ylim(1e-3, 0.98)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()


def plot_with_band(ax, x, flux, err, color, ls, lw=1, label=None):
    scaled = flux * x**energy_exponent
    scaled_err = err * x**energy_exponent
    ax.loglog(x, scaled, color=color, ls=ls, lw=lw, alpha=0.8, label=label)
    ax.fill_between(x, scaled - scaled_err, scaled + scaled_err, color=color, alpha=0.2)


for group, label, color in zip(groups, group_labels, group_colors, strict=False):
    plot_with_band(
        ax,
        energy,
        energy_fluxes["2017"][group],
        energy_errors["2017"][group],
        color,
        "--",
    )
    plot_with_band(
        ax,
        energy,
        energy_fluxes["2025"][group],
        energy_errors["2025"][group],
        color,
        "-",
        label=label,
    )

for v, ls in line_styles.items():
    plot_with_band(
        ax,
        energy,
        energy_fluxes[v]["Total"],
        energy_errors[v]["Total"],
        "black",
        ls,
        lw=2,
        label=f"Total {v}",
    )

ax.set_xlabel("Energy [GeV]")
ax.set_ylabel(f"Flux × $E^{{{energy_exponent}}}$")
ax.grid(True, alpha=0.3)
ax.legend(loc="lower center", ncol=2)
ax.set_xlim(energy[0], energy[-1])
ax.set_ylim(8.0, 9e4)
plt.tight_layout()
plt.show()

## Energy-per-nucleon flux comparison

Same three plots in the `GSFEnergyPerNucleon` view.

In [ ]:
fig, ax = plt.subplots()

for group, label, color in zip(groups, group_labels, group_colors, strict=False):
    for v in ("2017", "2025"):
        flux = nucleon_fluxes[v][group]
        ax.loglog(
            energy_per_nucleon,
            flux * energy_per_nucleon**energy_exponent,
            ls=line_styles[v],
            color=color,
            alpha=0.8,
            label=f"{label} ({v})" if v == "2025" else None,
        )

for v in ("2017", "2025"):
    flux = nucleon_fluxes[v]["Total"]
    ax.loglog(
        energy_per_nucleon,
        flux * energy_per_nucleon**energy_exponent,
        ls=line_styles[v],
        color="black",
        lw=2,
        alpha=0.9,
        label=f"Total ({v})",
    )

ax.set_xlabel("Energy per nucleon [GeV/nucleon]")
ax.set_ylabel(f"Nucleon flux × $E^{{{energy_exponent}}}$")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper right", ncol=2)
ax.set_xlim(energy_per_nucleon[0], energy_per_nucleon[-1])
ax.set_ylim(80.0, 9e4)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()

for group, label, color in zip(groups, group_labels, group_colors, strict=False):
    for v in ("2017", "2025"):
        rel = nucleon_errors[v][group] / nucleon_fluxes[v][group]
        ax.loglog(
            energy_per_nucleon,
            rel,
            ls=line_styles[v],
            color=color,
            alpha=0.8,
            label=f"{label} ({v})" if v == "2025" else None,
        )

for v in ("2017", "2025"):
    rel = nucleon_errors[v]["Total"] / nucleon_fluxes[v]["Total"]
    ax.loglog(
        energy_per_nucleon,
        rel,
        ls=line_styles[v],
        color="black",
        lw=2,
        alpha=0.9,
        label=f"Total ({v})",
    )

ax.set_xlabel("Energy per nucleon [GeV/nucleon]")
ax.set_ylabel("Relative error σ/Φ")
ax.grid(True, alpha=0.3)
ax.legend(loc="lower right", ncol=2)
ax.set_xlim(energy_per_nucleon[0], energy_per_nucleon[-1])
ax.set_ylim(3e-3, 1)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()

for group, label, color in zip(groups, group_labels, group_colors, strict=False):
    plot_with_band(
        ax,
        energy_per_nucleon,
        nucleon_fluxes["2017"][group],
        nucleon_errors["2017"][group],
        color,
        "--",
    )
    plot_with_band(
        ax,
        energy_per_nucleon,
        nucleon_fluxes["2025"][group],
        nucleon_errors["2025"][group],
        color,
        "-",
        label=label,
    )

for v, ls in line_styles.items():
    plot_with_band(
        ax,
        energy_per_nucleon,
        nucleon_fluxes[v]["Total"],
        nucleon_errors[v]["Total"],
        "black",
        ls,
        lw=2,
        label=f"Total {v}",
    )

ax.set_xlabel("Energy per nucleon [GeV/nucleon]")
ax.set_ylabel(f"Nucleon flux × $E^{{{energy_exponent}}}$")
ax.grid(True, alpha=0.3)
ax.legend(loc="lower center", ncol=2)
ax.set_xlim(energy_per_nucleon[0], energy_per_nucleon[-1])
ax.set_ylim(8.0, 9e4)
plt.tight_layout()
plt.show()

## Ratio of 2017 to 2025

Top: total nucleon flux ratio $\Phi_{2017}/\Phi_{2025}$ with the relative uncertainty of each version shaded. Bottom: ratio of absolute errors.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True)

flux_2017 = nucleon_fluxes["2017"]["Total"]
flux_2025 = nucleon_fluxes["2025"]["Total"]
err_2017 = nucleon_errors["2017"]["Total"]
err_2025 = nucleon_errors["2025"]["Total"]

flux_ratio = flux_2017 / flux_2025
err_ratio = err_2017 / err_2025

ax1.semilogx(energy_per_nucleon, flux_ratio, "b-", label="GSF2017")
ax1.fill_between(
    energy_per_nucleon,
    flux_ratio - err_2017 / flux_2017,
    flux_ratio + err_2017 / flux_2017,
    color="b",
    alpha=0.2,
)
ax1.fill_between(
    energy_per_nucleon,
    1 - err_2025 / flux_2025,
    1 + err_2025 / flux_2025,
    color="gray",
    alpha=0.2,
)
ax1.axhline(1, color="k", ls="--", alpha=0.5, label="GSF2025")
ax1.set_ylabel("Flux ratio (2017/2025)")
ax1.grid(True, alpha=0.3)
ax1.legend()
ax1.set_xlim(1.0, 3e9)
ax1.set_ylim(0.75, 1.25)

ax2.semilogx(energy_per_nucleon, err_ratio, "k-")
ax2.axhline(1, color="k", ls="--", alpha=0.5)
ax2.set_xlabel("Energy per nucleon [GeV/nucleon]")
ax2.set_ylabel("Error 2017 / Error 2025")
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0.0, 5.5)
plt.tight_layout()
plt.show()